> **Trust but verify.** Even examples Rick shares and AI-generated suggestions can be wrong or unsafe. Read code, commands and configuration before running them; use disposable copies, keep backups, and verify results. Don't let an example eat your homework.

# Follow a sentence through the model

This companion builds on the accepted [minimal notebook](minimal-pytorch.ipynb). Use the same [environment and setup](setup.md). Watch the input become tokens and tensors, inspect model stages, CPU placement and scores. You can watch without installing anything. The process, source-code and open-source project mapping is in [component-mapping.ipynb](component-mapping.ipynb).

The same pinned [BERT-tiny sentiment checkpoint](https://huggingface.co/M-FAC/bert-tiny-finetuned-sst2/tree/41ad6709ec46b414749b37daf49cf5ca1c7dba7c) loads locally on CPU. First-time loading downloads public files; sentence inference stays local. We inspect a forward pass with trained weights, not training.


In [ ]:
from html import escape
from IPython.display import HTML, display


def show_table(headers, rows):
    """Render small observations without adding a table package."""
    def row(values, tag):
        return "<tr>" + "".join(
            f"<{tag}>{escape(str(value))}</{tag}>" for value in values
        ) + "</tr>"
    display(HTML("<table>" + row(headers, "th")
                 + "".join(row(values, "td") for values in rows) + "</table>"))


def class_name(value):
    return f"{type(value).__module__}.{type(value).__name__}"


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "M-FAC/bert-tiny-finetuned-sst2"
REVISION = "41ad6709ec46b414749b37daf49cf5ca1c7dba7c"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, trust_remote_code=False
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, trust_remote_code=False, use_safetensors=False, weights_only=True
).to("cpu")
model.eval()
print("Model ready on", next(model.parameters()).device)


In [ ]:
show_table(["Model observation", "Value"], [
    ("Weights/tokenizer source", MODEL_ID),
    ("Pinned revision", REVISION),
    ("Model Python class", class_name(model)),
    ("Tokenizer Python class", class_name(tokenizer)),
    ("Model parameter device", str(next(model.parameters()).device)),
    ("Model evaluation mode", not model.training),
])


## 1. Encode the sentence

A token is a vocabulary piece, not always a whole word. `[CLS]` and `[SEP]` are special tokens added by this tokenizer. IDs select vocabulary entries; they are not sentiment scores.

This example truncates at 128 tokens, including special tokens. The display shows at most 16 positions and 200 input characters. The attention mask is 1 for included positions and 0 for padding; this single unpadded input has no padding. Token-type IDs identify a segment; this one-sentence input uses segment 0.


In [ ]:
text = "This movie was wonderful."
MAX_TOKENS = 128
original_count = len(tokenizer.encode(text, verbose=False))
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)

ids = inputs["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(ids)
print("Input:", text if len(text) <= 200 else text[:200] + "…")
print(f"Tokens including special tokens: {original_count}; used: {len(ids)}")
print("Truncated:", original_count > len(ids))
show_table(
    ["Position", "Token", "Vocabulary ID", "Attention mask", "Segment ID"],
    [(i, tokens[i], ids[i], int(inputs["attention_mask"][0, i]),
      int(inputs["token_type_ids"][0, i])) for i in range(min(16, len(ids)))],
)
if len(ids) > 16:
    print(f"Showing 16 of {len(ids)} token positions.")
show_table(
    ["Input tensor", "Shape", "Dtype", "Device"],
    [(name, tuple(value.shape), str(value.dtype), str(value.device))
     for name, value in inputs.items()],
)


## 2. Observe selected stages during one forward pass

The integer input IDs enter embeddings, producing a vector for each token. This checkpoint has two BERT encoder layers. The pooler transforms the first token's final representation into one vector per input. The classifier turns that vector into two raw scores (logits).

Shapes are `(batch, tokens, hidden features)` until pooling, then `(batch, features)` and `(batch, classes)`. We show the first four flattened values from each selected output, rounded for display. These are a small numerical sample, not a complete explanation of what the model learned.

[Forward hooks](https://docs.pytorch.org/docs/2.8/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook) observe outputs without replacing them. They are removed after each run, even if inference fails. Device labels report tensor placement; this is not hardware profiling or a speed comparison.


In [ ]:
stages = [("Embeddings", model.bert.embeddings)]
stages += [(f"Encoder layer {i + 1}", layer)
           for i, layer in enumerate(model.bert.encoder.layer)]
stages += [("Pooler", model.bert.pooler), ("Classifier", model.classifier)]
observations = []


def observe(name):
    def hook(module, arguments, output):
        value = output[0] if isinstance(output, tuple) else output
        sample = value.detach().flatten()[:4].cpu().tolist()
        observations.append((name, tuple(value.shape), str(value.dtype),
                             str(value.device), [round(x, 4) for x in sample]))
        # Returning None leaves the module output unchanged.
    return hook


handles = []
try:
    for name, module in stages:
        handles.append(module.register_forward_hook(observe(name)))
    with torch.inference_mode():
        logits = model(**inputs).logits
finally:
    for handle in handles:
        handle.remove()

show_table(["Stage", "Output shape", "Dtype", "Device", "First four values"],
           observations)


## 3. Turn raw scores into a prediction

Softmax turns the two logits into scores that sum to one. We choose the larger score. SST-2 class order is negative (0), positive (1); the checkpoint does not supply descriptive label names.

A larger score is the model's preference for that class, not proof of a person's feelings or a calibrated confidence estimate.


In [ ]:
scores = torch.softmax(logits, dim=-1)[0]
labels = ("negative", "positive")
prediction = labels[int(scores.argmax())]
show_table(
    ["Class", "Logit", "Softmax score"],
    [(label, round(float(logits[0, i]), 4), round(float(scores[i]), 4))
     for i, label in enumerate(labels)],
)
print("Input:", text if len(text) <= 200 else text[:200] + "…")
print("Prediction:", prediction)
print("Scores:", {label: round(float(score), 4) for label, score in zip(labels, scores)})


## Try another input, then reset

Change `text` in the encoding cell and rerun that cell and every cell below it. Compare token IDs, stage shapes, sample values and scores. Similar-length sentences can share shapes while their values and prediction change. A long input can demonstrate the 128-token limit without dumping hundreds of rows.

Use **Kernel → Restart Kernel and Run All Cells** for a fresh run. Rerunning the observation cell removes its hooks each time. Continue to [component-mapping.ipynb](component-mapping.ipynb) for the process, code and project map.
